# ST-GNN (BiLSTM+GAT) — offline training + seed-variance ensemble

Reproducible orchestrator for this branch's offline pipeline
(`main.py` -> `train`). Config is passed by **environment variables**
(the pipeline reads `WEATHER_SOURCE` / `FEATURE_SET` / `OPENMETEO_PATH` / `SEEDS` ...).

- **Single training run** -> logs the run to W&B.
- **Ensemble** -> a W&B sweep where the **only** swept parameter is the seed
  (fixed weather source / feature set), via `sweeps/sweep_agent.py`.
- `train()` uploads **no** artifact to W&B (metrics/summary only).

## 1. Setup

In [ ]:
import os, sys, subprocess, glob
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
os.environ['PYTHONPATH'] = str(REPO_ROOT) + os.pathsep + os.environ.get('PYTHONPATH', '')
os.chdir(REPO_ROOT)
print('repo root:', REPO_ROOT)

## 2. Shared config

One env-based config feeds both the single run and every sweep member.
Switch `weather_source` to `pvgis_legacy` (+ `feature_set='pvgis_legacy'`) to
train on the PVGIS weather instead of Open-Meteo.

In [ ]:
WEATHER = dict(
    weather_source='openmeteo_historical_forecast',  # or 'pvgis_legacy'
    feature_set='openmeteo_operational',             # or 'pvgis_legacy'
    openmeteo_path='data/openmeteo_piedmont_2019.nc',
)
TRAIN_ENV = dict(
    BILSTM_POOLING='last',
    QS_LOSS_WEIGHTING='0',
    QS_LOSS_FLOOR='0.2',
)

def make_env(seeds):
    """Environment for main.py: weather source + fixed knobs + the given seed(s)."""
    env = dict(os.environ)
    env['WEATHER_SOURCE'] = WEATHER['weather_source']
    env['FEATURE_SET'] = WEATHER['feature_set']
    env['OPENMETEO_PATH'] = WEATHER['openmeteo_path']
    env.update(TRAIN_ENV)
    env['SEEDS'] = ','.join(str(s) for s in seeds)
    return env

checks = {
    'main.py': Path('main.py').exists(),
    'sweep agent': Path('sweeps/sweep_agent.py').exists(),
    'openmeteo nc': (WEATHER['weather_source'] == 'pvgis_legacy'
                     or Path(WEATHER['openmeteo_path']).exists()),
}
for k, v in checks.items():
    print(('OK     ' if v else 'MISSING') + '  ' + k)

## 3. Single training run

Runs `main.py` for one seed. `train(use_wandb=True)` logs the run to W&B
(no artifact). Set `RUN_SINGLE = True` to actually launch.

In [ ]:
SEED = 42
single_env = make_env([SEED])
print('WEATHER_SOURCE =', single_env['WEATHER_SOURCE'])
print('FEATURE_SET    =', single_env['FEATURE_SET'])
print('OPENMETEO_PATH =', single_env['OPENMETEO_PATH'])
print('SEEDS          =', single_env['SEEDS'])
print('command        : python main.py   (config via env above; W&B on, no artifact)')

In [ ]:
RUN_SINGLE = False
if RUN_SINGLE:
    subprocess.run([sys.executable, 'main.py'], env=single_env, check=True)
else:
    print('RUN_SINGLE is False — not launching. Config above is what would run.')

## 4. Ensemble = W&B seed-only sweep

A real W&B sweep whose **only** parameter is `seed`; weather source / feature
set / pooling are fixed. Each worker (`sweeps/sweep_agent.py`) maps the config
to env vars and runs `main.py`. Mirrors
`sweeps/sweep_seed_variance_openmeteo.yaml`. No artifact upload.

In [ ]:
SEEDS = [42, 123, 2024, 7, 31337, 2718]

sweep_config = {
    'program': 'sweeps/sweep_agent.py',
    'method': 'grid',
    'metric': {'name': 'best_val_loss', 'goal': 'minimize'},
    'parameters': {
        'seed': {'values': SEEDS},          # the only thing that varies
        'bilstm_pooling': {'value': TRAIN_ENV['BILSTM_POOLING']},
        'weather_source': {'value': WEATHER['weather_source']},
        'feature_set': {'value': WEATHER['feature_set']},
        'openmeteo_path': {'value': WEATHER['openmeteo_path']},
    },
}
import json
print(json.dumps(sweep_config, indent=2))

In [ ]:
CREATE_SWEEP = False   # True -> register the sweep on W&B
RUN_AGENT = False      # True -> also run the agent in-process (blocks until done)
WANDB_PROJECT = 'PhysiQ-PV'
WANDB_ENTITY = 'albertopedalino-politecnico-di-torino'

if CREATE_SWEEP:
    import wandb
    sweep_id = wandb.sweep(sweep_config, project=WANDB_PROJECT, entity=WANDB_ENTITY)
    print('sweep_id  :', sweep_id)
    print('run with  : wandb agent ' + WANDB_ENTITY + '/' + WANDB_PROJECT + '/' + sweep_id)
    if RUN_AGENT:
        wandb.agent(sweep_id, project=WANDB_PROJECT, entity=WANDB_ENTITY, count=len(SEEDS))
else:
    print('Set CREATE_SWEEP=True to register. Or from a shell:')
    print('  wandb sweep sweeps/sweep_seed_variance_openmeteo.yaml')
    print('  wandb agent ' + WANDB_ENTITY + '/' + WANDB_PROJECT + '/<sweep_id>')

## 5. Results

Per-seed checkpoints written by the run(s).

In [ ]:
dirs = sorted(glob.glob('checkpoints/seq_len_24_pool*seed*'))
print('checkpoint dirs:', len(dirs))
for d in dirs:
    print(' ', d)